In [ ]:
!pip install sentencepiece datasets sacrebleu tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.2 MB/s eta 0:00:00


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from datasets import load_dataset
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
dataset = load_dataset("opus_books", "en-fr", split="train")
dataset = dataset.shuffle(seed=42).select(range(127085))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

en-fr/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/127085 [00:00<?, ? examples/s]

In [ ]:
import sentencepiece as spm

with open("train.txt", "w", encoding="utf-8") as f:
    for ex in dataset:
        f.write(ex["translation"]["en"] + "\n")
        f.write(ex["translation"]["fr"] + "\n")


In [ ]:
spm.SentencePieceTrainer.train(
    input="train.txt",
    model_prefix="spm",
    vocab_size=16000,
    model_type="bpe",
    pad_id=0,
    bos_id=1,
    eos_id=2,
    unk_id=3
)

sp = spm.SentencePieceProcessor()
sp.load("spm.model")

PAD, BOS, EOS = 0, 1, 2
VOCAB_SIZE = sp.get_piece_size()


In [ ]:
MAX_LEN = 128
BATCH_SIZE = 16

class TranslationDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, sp):
        self.dataset = dataset
        self.sp = sp

    def __len__(self):
        return len(self.dataset)

    def encode(self, text):
        ids = [BOS] + self.sp.encode(text, out_type=int)[:MAX_LEN-2] + [EOS]
        return torch.tensor(ids)

    def __getitem__(self, idx):
        ex = self.dataset[idx]
        src = self.encode(ex["translation"]["en"])
        tgt = self.encode(ex["translation"]["fr"])
        return src, tgt

def collate(batch):
    src = nn.utils.rnn.pad_sequence([x[0] for x in batch], batch_first=True, padding_value=PAD)
    tgt = nn.utils.rnn.pad_sequence([x[1] for x in batch], batch_first=True, padding_value=PAD)
    return src.to(device), tgt.to(device)

train_loader = DataLoader(
    TranslationDataset(dataset, sp),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate,
    num_workers=0
)



In [ ]:
def make_src_mask(src):
    return (src != PAD).unsqueeze(1).unsqueeze(2)

def make_tgt_mask(tgt):
    B, T = tgt.size()
    pad_mask = (tgt != PAD).unsqueeze(1).unsqueeze(2)
    causal_mask = torch.tril(torch.ones((T, T), device=tgt.device)).bool().unsqueeze(0).unsqueeze(0)
    return pad_mask & causal_mask


# Transformer

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        assert d_model % heads == 0
        self.d_k = d_model // heads
        self.h = heads
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V, mask=None):
        B = Q.size(0)
        Q = self.q(Q).view(B, -1, self.h, self.d_k).transpose(1,2)
        K = self.k(K).view(B, -1, self.h, self.d_k).transpose(1,2)
        V = self.v(V).view(B, -1, self.h, self.d_k).transpose(1,2)

        scores = (Q @ K.transpose(-2,-1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, torch.finfo(scores.dtype).min)
        attn = F.softmax(scores, dim=-1)
        out = attn @ V
        out = out.transpose(1,2).contiguous().view(B, -1, self.h*self.d_k)
        return self.out(out)


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
    def forward(self, x):
        return self.net(x)


In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, heads, d_ff):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    def forward(self, x, mask):
        x = self.norm1(x + self.attn(x, x, x, mask))
        x = self.norm2(x + self.ffn(x))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, heads, d_ff):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, heads)
        self.cross_attn = MultiHeadAttention(d_model, heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
    def forward(self, x, enc, src_mask, tgt_mask):
        x = self.norm1(x + self.self_attn(x, x, x, tgt_mask))
        x = self.norm2(x + self.cross_attn(x, enc, enc, src_mask))
        x = self.norm3(x + self.ffn(x))
        return x


In [ ]:
class Transformer(nn.Module):
    def __init__(self, vocab, d_model=384, layers=4, heads=6, d_ff=1536, max_len=512):
        super().__init__()
        self.embed = nn.Embedding(vocab, d_model)
        self.pos = nn.Embedding(max_len, d_model)
        self.enc = nn.ModuleList([EncoderLayer(d_model, heads, d_ff) for _ in range(layers)])
        self.dec = nn.ModuleList([DecoderLayer(d_model, heads, d_ff) for _ in range(layers)])
        self.fc = nn.Linear(d_model, vocab)

    def forward(self, src, tgt, src_mask, tgt_mask):
        B, S = src.size()
        T = tgt.size(1)
        src = self.embed(src) + self.pos(torch.arange(S, device=src.device))
        tgt = self.embed(tgt) + self.pos(torch.arange(T, device=tgt.device))
        for l in self.enc: src = l(src, src_mask)
        for l in self.dec: tgt = l(tgt, src, src_mask, tgt_mask)
        return self.fc(tgt)


In [ ]:
model = Transformer(VOCAB_SIZE).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
scaler = GradScaler()


/tmp/ipykernel_1020/1100359743.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [ ]:
def train_epoch():
    model.train()
    total = 0
    for src, tgt in tqdm(train_loader):
        src_mask = make_src_mask(src)
        tgt_mask = make_tgt_mask(tgt[:, :-1])
        with autocast():
            out = model(src, tgt[:, :-1], src_mask, tgt_mask)
            loss = criterion(out.view(-1, VOCAB_SIZE), tgt[:, 1:].reshape(-1))
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        total += loss.item()
    return total / len(train_loader)




In [ ]:
EPOCHS = 200
for epoch in range(EPOCHS):
    loss = train_epoch()
    print(f"Epoch {epoch+1} | Loss {loss:.3f}")


    if (epoch + 1) % 10 == 0:
        checkpoint_path = f"checkpoint_epoch_{epoch+1}.pt"
        torch.save(model, checkpoint_path)
        print(f"Full model saved at {checkpoint_path}")



  0%|          | 0/7943 [00:00<?, ?it/s]/tmp/ipykernel_1020/3750262109.py:7: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 7943/7943 [07:48<00:00, 16.97it/s]


Epoch 1 | Loss 3.948


100%|██████████| 7943/7943 [07:48<00:00, 16.95it/s]


Epoch 2 | Loss 2.879


100%|██████████| 7943/7943 [07:48<00:00, 16.96it/s]


Epoch 3 | Loss 2.415


  4%|▍         | 300/7943 [00:17<08:29, 15.02it/s]

In [ ]:
model = Transformer(VOCAB_SIZE).to(device)
model.load_state_dict(torch.load("/content/Tom_En_Fr_model.pt", map_location=torch.device('cpu')))
model.eval()

In [ ]:
@torch.no_grad()
def translate_greedy(sentence, max_len=60):
    model.eval()

    # encode source
    src_ids = [BOS] + sp.encode(sentence, out_type=int) + [EOS]
    src = torch.tensor(src_ids).unsqueeze(0).to(device)

    src_mask = make_src_mask(src)

    # encoder
    enc = model.embed(src) + model.pos(
        torch.arange(src.size(1), device=device)
    )
    for layer in model.enc:
        enc = layer(enc, src_mask)

    # decoder start
    tgt = torch.tensor([[BOS]], device=device)

    for _ in range(max_len):
        tgt_mask = make_tgt_mask(tgt)

        dec = model.embed(tgt) + model.pos(
            torch.arange(tgt.size(1), device=device)
        )
        for layer in model.dec:
            dec = layer(dec, enc, src_mask, tgt_mask)

        next_token = model.fc(dec[:, -1]).argmax(-1).item()
        tgt = torch.cat([tgt, torch.tensor([[next_token]], device=device)], dim=1)

        if next_token == EOS:
            break

    return sp.decode(tgt[0].tolist())


In [ ]:
test_sentence = "Hello There Where in Paris do you live."
print("SRC:", test_sentence)
print("PRED:", translate_greedy(test_sentence))


In [ ]:
def quick_eval(n=5):
    model.eval()
    for i in range(n):
        src, tgt = train_loader.dataset[i]
        src_text = sp.decode(src.tolist())
        tgt_text = sp.decode(tgt.tolist())

        pred = translate_greedy(src_text)

        print("-" * 50)
        print("SRC:", src_text)
        print("GT :", tgt_text)
        print("PRD:", pred)


In [ ]:
quick_eval(5)


In [ ]:
import sacrebleu

@torch.no_grad()
def bleu_score(samples=200):
    refs, hyps = [], []

    for i in range(samples):
        src, tgt = train_loader.dataset[i]
        src_text = sp.decode(src.tolist())
        tgt_text = sp.decode(tgt.tolist())

        pred = translate_greedy(src_text)

        refs.append([tgt_text])
        hyps.append(pred)

    bleu = sacrebleu.corpus_bleu(hyps, refs)
    return bleu.score


print("BLEU:", bleu_score(100))


In [ ]:
torch.save(model.state_dict(), "Tom_En_Fr_model.pt")

In [ ]:
bleu = bleu_score(100)
if bleu > 90:
  torch.save(model.state_dict(), "Tom_En_Fr_model.pt")
  print("Model saved")